# Phase 4 Visual Audit: DuckDB Analytical Marts, Event Studies & DiD

This thin visual audit notebook tests the analytical and econometric layer:
1. **DuckDB In-Process Engine:** Executes high-performance SQL directly over Parquet files without database overhead.
2. **Monthly & Quarterly Marts:** Aggregates multi-tier sentiment, net sentiment ratios, and volume.
3. **Event Studies (T-3 to T+3):** Quantifies morale shifts around leadership ultimatums (Jassy, Musk, Zuckerberg, Pichai).
4. **Difference-in-Differences (DiD):** Evaluates treated firms (mass layoffs >= 1,000) vs resilient controls (Apple, Airbnb, etc.).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on Python sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import duckdb
from src.common.config import EngineConfig
from src.common.azure_storage import StorageManager
from src.analytics.duckdb_marts import DuckDBMartsManager
from src.analytics.event_study import EventStudyAnalyzer
from src.analytics.did_analysis import DiDAnalyzer

config = EngineConfig.load()
storage = StorageManager(config)
marts = DuckDBMartsManager(config)
print("✅ DuckDB Analytics engine initialized successfully.")

### 1. Register Parquet Views in DuckDB

In [ ]:
# Register stored Parquet tables if present or load sample views
marts.register_tables()

# Inspect tables and views
display(marts.query("SHOW TABLES"))

### 2. Company Monthly Morale Trajectory Query

In [ ]:
df_monthly = marts.build_monthly_mart()
print(f"Total monthly company records aggregated: {len(df_monthly)}")
display(df_monthly.head(10))

### 3. Leadership Controversy & Event Study Analysis (T-3 to T+3)

In [ ]:
event_analyzer = EventStudyAnalyzer(marts_manager=marts, config=config)
df_events = event_analyzer.run_all_events(save=False)
print(f"Analyzed {len(df_events)} registered leadership controversy events.")
display(df_events[["event_key", "company", "leader", "anchor_month", "pre_records", "post_records", "delta_roberta", "volume_change_pct"]])

### 4. 2022–2026 Restructuring Difference-in-Differences (DiD)

In [ ]:
did_analyzer = DiDAnalyzer(marts_manager=marts, config=config)
did_results = did_analyzer.calculate_did_estimator(score_col="avg_roberta")

print("Difference-in-Differences Results:")
print(f"  Treated Pre -> Post Delta: {did_results.get('delta_treated')}")
print(f"  Control Pre -> Post Delta: {did_results.get('delta_control')}")
print(f"  DiD Estimator:             {did_results.get('did_post_vs_pre')}")
print(f"  Shock Phase DiD:           {did_results.get('shock_did')}")

display(pd.DataFrame(did_results.get('cohort_matrix', {})))